In [3]:
import json
from dataclasses import dataclass, field
from typing import List, Optional

# ====================== БАЗА ДАННЫХ ======================
DB_JSON = """
[
  {"type": "книга", "title": "Война и мир", "author": "Толстой", "year": 1869, "pages": 1225, "publisher": "АСТ", "language": "русский", "topic": "литература"},
  {"type": "книга", "title": "Преступление и наказание", "author": "Достоевский", "year": 1866, "pages": 671, "publisher": "Эксмо", "language": "русский", "topic": "литература"},
  {"type": "книга", "title": "Мастер и Маргарита", "author": "Булгаков", "year": 1967, "pages": 480, "publisher": "АСТ", "language": "русский", "topic": "литература"},
  {"type": "статья", "title": "Современные методы обработки языка", "author": "Иванов", "year": 2020, "pages": 15, "publisher": "Питер", "language": "русский", "topic": "лингвистика"},
  {"type": "статья", "title": "Deep Learning for NLP", "author": "Smith", "year": 2021, "pages": 22, "publisher": "Williams", "language": "английский", "topic": "программирование"},
  {"type": "журнал", "title": "Программист", "author": null, "year": 2023, "pages": 96, "publisher": "Питер", "language": "русский", "topic": "программирование"},
  {"type": "книга", "title": "Лолита", "author": "Набоков", "year": 1955, "pages": 336, "publisher": "Эксмо", "language": "русский", "topic": "литература"}
]
"""

def load_database():
    return json.loads(DB_JSON)

print(f"База данных загружена, записей: {len(load_database())}")

# ====================== ТОКЕНЫ И ДЕРЕВО ======================
@dataclass
class Token:
    type: str
    value: str
    pos: int

@dataclass
class Node:
    name: str
    children: List['Node'] = field(default_factory=list)
    value: Optional[str] = None

    def add(self, child: Optional['Node']):
        if child is not None:
            self.children.append(child)
        return self

    def pretty(self, prefix="", is_last=True):
        """Красивое дерево с символами"""
        connector = "└── " if is_last else "├── "
        line = prefix + connector + self.name
        if self.value is not None:
            line += f": «{self.value}»"

        lines = [line]

        if self.children:
            new_prefix = prefix + ("    " if is_last else "│   ")
            for i, child in enumerate(self.children):
                last = (i == len(self.children) - 1)
                lines.append(child.pretty(new_prefix, last).rstrip("\n"))

        return "\n".join(lines)

    # Чтобы Jupyter не выводил уродливый repr
    def __repr__(self):
        return self.pretty()

# ====================== ОШИБКИ ======================
class ParseError(Exception):
    def __init__(self, message: str, token: Optional[Token] = None, expected: str = None, hint: str = None):
        self.message = message
        self.token = token
        self.expected = expected
        self.hint = hint

    def full_message(self) -> str:
        parts = [f"ОШИБКА: {self.message}"]
        if self.token:
            parts.append(f"  → Проблемное место: «{self.token.value}» (позиция {self.token.pos})")
        if self.expected:
            parts.append(f"  → Ожидалось: {self.expected}")
        if self.hint:
            parts.append(f"  → Как исправить: {self.hint}")
        return "\n".join(parts)

# ====================== СЛОВАРИ ======================
ACTIONS    = {"найди", "покажи", "выведи"}
OBJECTS    = {"книги", "статьи", "журналы", "издания"}
AUTHORS    = {"толстого", "достоевского", "пушкина", "чехова", "булгакова", "набокова"}
TOPICS     = {"программированию", "лингвистике", "математике", "физике", "истории", "философии", "литературе"}
PUBLISHERS = {"аст", "эксмо", "питер", "вильямс", "манн", "иванов"}
LANGUAGES  = {"русском", "английском", "немецком", "французском"}

# ====================== ЛЕКСЕР ======================
def tokenize(text: str) -> List[Token]:
    tokens = []
    pos = 0
    for raw in text.split():
        word = raw.lower().strip(".,!?;:\"'")
        if not word:
            pos += len(raw) + 1
            continue
        token_type = "NUMBER" if word.isdigit() else "WORD"
        tokens.append(Token(token_type, word, pos))
        pos += len(raw) + 1
    return tokens

# ====================== ПАРСЕР ======================
class Parser:
    def __init__(self, tokens: List[Token]):
        self.tokens = tokens
        self.pos = 0

    def current(self) -> Optional[Token]:
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def advance(self):
        self.pos += 1

    def match(self, *values):
        tok = self.current()
        if tok and tok.value in values:
            self.advance()
            return tok
        return None

    def expect(self, *values, error_msg=None, hint=None):
        tok = self.match(*values)
        if tok:
            return tok
        expected = " или ".join(f"«{v}»" for v in values)
        raise ParseError(error_msg or "Неожиданный токен", self.current(), expected, hint)

    def expect_number(self, hint=None):
        tok = self.current()
        if tok and tok.type == "NUMBER":
            self.advance()
            return tok
        raise ParseError("Ожидалось число", tok, "число (например 2020 или 500)", hint)

    def parse(self):
        tree = self.parse_query()
        if self.current() is not None:
            raise ParseError(
                "После конца запроса остались лишние слова",
                self.current(),
                "конец запроса",
                "удалите лишние слова в конце"
            )
        return tree

    def parse_query(self):
        node = Node("Query")
        node.add(self.parse_action())
        node.add(self.parse_object())
        node.add(self.parse_specs())
        return node

    def parse_action(self):
        tok = self.match(*ACTIONS)
        if not tok:
            raise ParseError(
                "Запрос должен начинаться с действия",
                self.current(),
                "«найди», «покажи» или «выведи»",
                "начните запрос с одного из этих слов"
            )
        action = tok.value
        if self.match("все"):
            action += " все"
        return Node("Action", value=action)

    def parse_object(self):
        tok = self.expect(
            *OBJECTS,
            error_msg="Ожидался тип объекта поиска",
            hint="укажите: книги, статьи, журналы или издания"
        )
        return Node("Object", value=tok.value)

    def parse_specs(self):
        node = Node("Specs")
        while True:
            spec = self.try_parse_spec()
            if spec is None:
                break
            node.add(spec)
        return node

    def try_parse_spec(self):
        tok = self.current()
        if tok is None:
            return None

        if tok.value in AUTHORS:
            return self.parse_author_spec()
        if tok.value == "изданные" or tok.type == "NUMBER":
            return self.parse_year_spec()
        if tok.value == "по":
            return self.parse_topic_spec()
        if tok.value in ("объёмом", "от") or (tok.type == "NUMBER" and self._looks_like_pages()):
            return self.parse_pages_spec()
        if tok.value in ("издательства", "издательство"):
            return self.parse_publisher_spec()
        if tok.value == "на":
            return self.parse_language_spec()
        return None

    def _looks_like_pages(self):
        return (self.pos + 1 < len(self.tokens) and
                self.tokens[self.pos + 1].value == "страниц")

    def parse_author_spec(self):
        node = Node("AuthorSpec")
        authors = [self.expect(*AUTHORS).value.capitalize()]
        while self.match("и"):
            authors.append(
                self.expect(
                    *AUTHORS,
                    error_msg="После «и» ожидался автор",
                    hint="укажите фамилию автора в родительном падеже"
                ).value.capitalize()
            )
        node.value = " и ".join(authors)
        return node

    def parse_year_spec(self):
        node = Node("YearSpec")
        if self.match("изданные"):
            if self.match("после"):
                year = self.expect_number("укажите год после слова «после»")
                node.value = f"изданные после {year.value}"
            elif self.match("до"):
                year = self.expect_number("укажите год после слова «до»")
                node.value = f"изданные до {year.value}"
            elif self.match("в"):
                year = self.expect_number()
                self.expect("года", error_msg="После года ожидалось слово «года»")
                node.value = f"изданные в {year.value} года"
            else:
                year = self.expect_number()
                self.expect("года")
                node.value = f"изданные {year.value} года"
        else:
            year = self.expect_number()
            self.expect(
                "года",
                error_msg="После числа ожидалось слово «года»",
                hint="напишите, например: 2020 года"
            )
            node.value = f"{year.value} года"
        return node

    def parse_topic_spec(self):
        self.expect("по")
        topic = self.expect(
            *TOPICS,
            error_msg="Ожидалась тема",
            hint="например: по программированию, по лингвистике"
        )
        return Node("TopicSpec", value=f"по {topic.value}")

    def parse_pages_spec(self):
        node = Node("PagesSpec")
        if self.match("объёмом"):
            num = self.expect_number("после «объёмом» укажите число страниц")
            self.expect("страниц", error_msg="После числа ожидалось слово «страниц»")
            node.value = f"объёмом {num.value} страниц"
        elif self.match("от"):
            n1 = self.expect_number()
            self.expect("до")
            n2 = self.expect_number()
            self.expect("страниц")
            node.value = f"от {n1.value} до {n2.value} страниц"
        else:
            num = self.expect_number()
            self.expect("страниц")
            node.value = f"{num.value} страниц"
        return node

    def parse_publisher_spec(self):
        self.match("издательства") or self.match("издательство")
        pub = self.expect(
            *PUBLISHERS,
            error_msg="Ожидалось название издательства",
            hint="например: АСТ, Эксмо, Питер, Вильямс"
        )
        name = "Иванов и Фербер" if pub.value == "иванов" else pub.value.upper()
        return Node("PublisherSpec", value=f"издательства {name}")

    def parse_language_spec(self):
        self.expect("на")
        lang = self.expect(
            *LANGUAGES,
            error_msg="Ожидался язык",
            hint="например: на русском языке"
        )
        self.expect("языке", error_msg="После языка ожидалось слово «языке»")
        return Node("LanguageSpec", value=f"на {lang.value} языке")

# ====================== ГЛАВНАЯ ФУНКЦИЯ ======================
def analyze(query: str):
    """Разбирает один запрос и красиво выводит результат"""
    print("=" * 70)
    print("Запрос:", query)
    print("-" * 70)

    try:
        tokens = tokenize(query)
        if not tokens:
            print("Результат: ОШИБКА")
            print("Пустой запрос")
            return

        parser = Parser(tokens)
        tree = parser.parse()

        print("Результат: УСПЕХ ✓")
        print("\nДерево разбора:")
        print(tree.pretty())
        # return убираем, чтобы Jupyter ничего лишнего не печатал

    except ParseError as e:
        print("Результат: ОШИБКА ✗")
        print(e.full_message())

База данных загружена, записей: 7


In [4]:
# Успешные запросы
analyze("найди все книги")
analyze("покажи статьи 2020 года")
analyze("найди книги Толстого и Достоевского")
analyze("выведи все журналы по программированию")
analyze("найди книги по лингвистике изданные после 2015")
analyze("найди книги Булгакова объёмом 480 страниц издательства АСТ")
analyze("покажи статьи на английском языке")

Запрос: найди все книги
----------------------------------------------------------------------
Результат: УСПЕХ ✓

Дерево разбора:
└── Query
    ├── Action: «найди все»
    ├── Object: «книги»
    └── Specs
Запрос: покажи статьи 2020 года
----------------------------------------------------------------------
Результат: УСПЕХ ✓

Дерево разбора:
└── Query
    ├── Action: «покажи»
    ├── Object: «статьи»
    └── Specs
        └── YearSpec: «2020 года»
Запрос: найди книги Толстого и Достоевского
----------------------------------------------------------------------
Результат: УСПЕХ ✓

Дерево разбора:
└── Query
    ├── Action: «найди»
    ├── Object: «книги»
    └── Specs
        └── AuthorSpec: «Толстого и Достоевского»
Запрос: выведи все журналы по программированию
----------------------------------------------------------------------
Результат: УСПЕХ ✓

Дерево разбора:
└── Query
    ├── Action: «выведи все»
    ├── Object: «журналы»
    └── Specs
        └── TopicSpec: «по программирова

In [5]:
analyze("найди книги Толстого и")
analyze("найди книги изданные после")
analyze("найди книги объёмом 500")
analyze("найди книги объёмом страниц")
analyze("покажи статьи 2020")
analyze("книги Толстого")
analyze("найди все")

Запрос: найди книги Толстого и
----------------------------------------------------------------------
Результат: ОШИБКА ✗
ОШИБКА: После «и» ожидался автор
  → Ожидалось: «чехова» или «толстого» или «достоевского» или «пушкина» или «булгакова» или «набокова»
  → Как исправить: укажите фамилию автора в родительном падеже
Запрос: найди книги изданные после
----------------------------------------------------------------------
Результат: ОШИБКА ✗
ОШИБКА: Ожидалось число
  → Ожидалось: число (например 2020 или 500)
  → Как исправить: укажите год после слова «после»
Запрос: найди книги объёмом 500
----------------------------------------------------------------------
Результат: ОШИБКА ✗
ОШИБКА: После числа ожидалось слово «страниц»
  → Ожидалось: «страниц»
Запрос: найди книги объёмом страниц
----------------------------------------------------------------------
Результат: ОШИБКА ✗
ОШИБКА: Ожидалось число
  → Проблемное место: «страниц» (позиция 20)
  → Ожидалось: число (например 2020 или 500

In [ ]:
# Интерактивный режим
while True:
    q = input("Запрос (или 'выход')> ").strip()
    if not q or q.lower() in ("выход", "exit", "quit"):
        print("Выход.")
        break
    analyze(q)